
# EfficientNet-B0 Training ( Without SAM )


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define Dataset Path
data_dir = '/content/drive/MyDrive/kaggle_dataset_train'

# Hyperparameters
img_size = (224, 224)  # Image size for EfficientNet-B0
batch_size = 32
epochs = 20
learning_rate = 0.001

# Necessary preprocessing steps
def custom_preprocessing(image):
    image = tf.image.resize(image, img_size)  # Resize image
    image = tf.image.convert_image_dtype(image, tf.float32)  # Convert to float32
    image = tf.keras.applications.efficientnet.preprocess_input(image)  # EfficientNet preprocessing
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)  # Adjust contrast
    image = tf.image.random_saturation(image, lower=0.9, upper=1.1)  # Adjust saturation
    image = tf.image.random_brightness(image, max_delta=0.2)  # Adjust brightness
    image = tf.image.random_flip_left_right(image)  # Random horizontal flip
    image = tf.image.random_flip_up_down(image)  # Random vertical flip
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.01, dtype=tf.float32)  # Add Gaussian noise
    return image

# Step 3: Data Augmentation (Custom)
datagen = ImageDataGenerator(
    rotation_range=15,               # Random rotation up to 15 degrees
    shear_range=0.2,                 # Apply shear transformation
    brightness_range=[0.8, 1.2],     # Adjust brightness between 80% and 120%
    preprocessing_function=custom_preprocessing,
    horizontal_flip=True,            # Random horizontal flip
    vertical_flip=True,              # Random vertical flip
    zoom_range=[0.8, 1.2],           # Random zoom up to 20%
    validation_split=0.2             # Use 20% of data for validation
)

# Step 4: Data Generators
train_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# Step 5: Build EfficientNet-B0 Model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers initially
base_model.trainable = False

# Add custom layers for classification
x = base_model.output
x = GlobalAveragePooling2D()(x)  # Pooling layer
x = Dropout(0.5)(x)              # Regularization
x = Dense(128, activation='relu', kernel_initializer='he_normal')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # Binary classification output

model = Model(inputs=base_model.input, outputs=output)

# Step 6: Compile the Model
model.compile(optimizer=Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Step 7: Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

# Custom Callback for Metrics
class MetricsCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        val_pred = (self.model.predict(val_generator) > 0.5).astype(int).flatten()
        val_true = val_generator.classes
        print("\nClassification Report:")
        print(classification_report(val_true, val_pred, target_names=['Benign', 'Malignant']))
        roc_auc = roc_auc_score(val_true, self.model.predict(val_generator).flatten())
        print(f"ROC-AUC Score: {roc_auc:.4f}")

metrics_callback = MetricsCallback()

# Step 8: Train the Model (Initial Phase)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr, metrics_callback]
)

# Step 9: Unfreeze Top 100 Layers
for layer in base_model.layers[-100:]:
    layer.trainable = True

# Recompile the model with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=learning_rate / 10),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Step 10: Fine-Tune the Model
history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr, metrics_callback]
)

# Step 11: Save the Model
model.save('/content/drive/MyDrive/kaggledatasetmodel.h5')

# Step 12: Evaluate the Model
loss, accuracy = model.evaluate(val_generator)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


17/17 ━━━━━━━━━━━━━━━━━━━━ 78s 4s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.52      0.59      0.55       288
   Malignant       0.42      0.35      0.38       239

    accuracy                           0.48       527
   macro avg       0.47      0.47      0.47       527
weighted avg       0.47      0.48      0.48       527

17/17 ━━━━━━━━━━━━━━━━━━━━ 67s 4s/step
ROC-AUC Score: 0.4945
66/66 ━━━━━━━━━━━━━━━━━━━━ 654s 9s/step - accuracy: 0.7200 - loss: 0.5583 - val_accuracy: 0.7913 - val_loss: 0.4453 - learning_rate: 0.0010
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.54      0.55      0.54       288
   Malignant       0.44      0.43      0.43       239

    accuracy                           0.50       527
   macro avg       0.49      0.49      0.49       527
weighted avg       0.49      0.50      0.49       527

17/17 ━━

17/17 ━━━━━━━━━━━━━━━━━━━━ 51s 3s/step - accuracy: 0.8666 - loss: 0.3010
Validation Accuracy: 85.58%


## Testing the model[EfficientNet B0 without SAM]

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

drive.mount('/content/drive')


# Step 1: Load the trained model
model_path = '/content/drive/MyDrive/kaggledatasetmodel.h5'
model = load_model(model_path)

# Step 2: Define the test data directory and generator
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'  # Replace with your test folder path

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input  # Use same preprocessing as training
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Do not shuffle to maintain the order of predictions
)

# Step 3: Predict on the test dataset
test_predictions = model.predict(test_generator)
test_predictions = (test_predictions > 0.5).astype(int).flatten()

# Step 4: Evaluate the model's performance
test_true = test_generator.classes  # True labels from the test dataset

# Classification Report
print("\nClassification Report:")
print(classification_report(test_true, test_predictions, target_names=['Benign', 'Malignant']))

# ROC-AUC Score
roc_auc = roc_auc_score(test_true, model.predict(test_generator).flatten())
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Step 5: Displaying Overall Accuracy
accuracy = np.mean(test_predictions == test_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Found 660 images belonging to 2 classes.


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


21/21 ━━━━━━━━━━━━━━━━━━━━ 122s 6s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.93      0.87      0.90       360
   Malignant       0.85      0.92      0.88       300

    accuracy                           0.89       660
   macro avg       0.89      0.89      0.89       660
weighted avg       0.89      0.89      0.89       660

21/21 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step
ROC-AUC Score: 0.9617
Test Accuracy: 88.94%


# Adding a spatial attention module  

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, Multiply, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from google.colab import drive

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define Dataset Path
data_dir = '/content/drive/MyDrive/kaggle_dataset_train'

# Hyperparameters
img_size = (224, 224)  # Image size for EfficientNet-B0
batch_size = 32
epochs = 20
learning_rate = 0.001

def custom_preprocessing(image):
    image = tf.image.resize(image, img_size)  # Resize image
    image = tf.image.convert_image_dtype(image, tf.float32)  # Convert to float32
    image = tf.keras.applications.efficientnet.preprocess_input(image)  # EfficientNet preprocessing
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)  # Adjust contrast
    image = tf.image.random_saturation(image, lower=0.9, upper=1.1)  # Adjust saturation
    image = tf.image.random_brightness(image, max_delta=0.2)  # Adjust brightness
    image = tf.image.random_flip_left_right(image)  # Random horizontal flip
    image = tf.image.random_flip_up_down(image)  # Random vertical flip
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.01, dtype=tf.float32)  # Add Gaussian noise
    return image

# Step 3: Data Augmentation (Custom)
datagen = ImageDataGenerator(
    rotation_range=15,               # Random rotation up to 15 degrees
    shear_range=0.2,                 # Apply shear transformation
    brightness_range=[0.8, 1.2],     # Adjust brightness between 80% and 120%
    preprocessing_function=custom_preprocessing,
    horizontal_flip=True,            # Random horizontal flip
    vertical_flip=True,              # Random vertical flip
    zoom_range=[0.8, 1.2],           # Random zoom up to 20%
    validation_split=0.2             # Use 20% of data for validation
)

# Step 4: Data Generators
train_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# Define Spatial Attention Module (SAM) with Dropout and Regularization
def spatial_attention_module(feature_map):
    # Attention map with dropout and L2 regularization
    attention_map = Conv2D(1, kernel_size=(3, 3), activation='sigmoid', padding='same',
                           kernel_regularizer=l2(1e-4))(feature_map)
    attention_map = Dropout(0.3)(attention_map)

    # Multiply feature map by attention map
    enhanced_feature_map = Multiply()([feature_map, attention_map])

    # Optionally add residual connection
    output = Add()([feature_map, enhanced_feature_map])
    return output

# Step 5: Build EfficientNet-B0 Model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers initially
base_model.trainable = False

# Extract feature maps after Block 5
block5_output = base_model.get_layer('block5a_activation').output

# Add Spatial Attention Module to Block 5 output
x = spatial_attention_module(block5_output)

# Add custom classification layers
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)  # Regularization
x = Dense(128, activation='relu', kernel_initializer='he_normal')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # Binary classification output

# Build the model
model = Model(inputs=base_model.input, outputs=output)

# Step 6: Compile the Model
model.compile(optimizer=Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Step 7: Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

# Step 8: Train the Model (Initial Phase)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr]
)

# Step 9: Unfreeze Top 100 Layers
for layer in base_model.layers[-100:]:
    layer.trainable = True

# Recompile the model with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=learning_rate / 10),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Step 10: Fine-Tune the Model
history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr]
)

# Step 11: Save the Model
model.save('/content/drive/MyDrive/SAMv2_kaggledatasetmodel.h5')

# Step 12: Evaluate the Model
loss, accuracy = model.evaluate(val_generator)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

# Step 13: Visualize Attention Maps
def visualize_attention(sample_input):
    feature_map_model = Model(inputs=base_model.input, outputs=model.get_layer('add').output)
    feature_map = feature_map_model.predict(sample_input)
    attention_map = feature_map[0, :, :, 0]  # Extract the first attention map

    # Plot original image and attention map
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(sample_input[0])
    plt.subplot(1, 2, 2)
    plt.title("Attention Map")
    plt.imshow(attention_map, cmap='viridis')
    plt.show()

# Test the attention map visualization
sample_input, _ = val_generator.next()  # Get a batch of validation data
visualize_attention(sample_input)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


66/66 ━━━━━━━━━━━━━━━━━━━━ 489s 7s/step - accuracy: 0.5683 - loss: 0.7693 - val_accuracy: 0.7533 - val_loss: 0.4909 - learning_rate: 0.0010
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 248s 4s/step - accuracy: 0.7381 - loss: 0.5280 - val_accuracy: 0.8140 - val_loss: 0.4373 - learning_rate: 0.0010
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 269s 4s/step - accuracy: 0.7599 - loss: 0.4831 - val_accuracy: 0.7856 - val_loss: 0.4448 - learning_rate: 0.0010
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 285s 3s/step - accuracy: 0.7944 - loss: 0.4418 - val_accuracy: 0.8197 - val_loss: 0.4341 - learning_rate: 0.0010
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 228s 3s/step - accuracy: 0.8023 - loss: 0.4497 - val_accuracy: 0.7761 - val_loss: 0.4385 - learning_rate: 0.0010
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 308s 4s/step - accuracy: 0.7998 - loss: 0.4202 - val_accuracy: 0.8046 - val_loss: 0.4260 - learning_rate: 0.0010
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 237s 3s/step - accuracy: 0.8110 - loss: 0.4019 - val_accuracy: 0.76

17/17 ━━━━━━━━━━━━━━━━━━━━ 46s 3s/step - accuracy: 0.8327 - loss: 0.4062
Validation Accuracy: 82.35%


AttributeError: 'DirectoryIterator' object has no attribute 'next'

# Testing Model with SAM

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

drive.mount('/content/drive')


# Step 1: Load the trained model
model_path = '/content/drive/MyDrive/SAMv2_kaggledatasetmodel.h5'
model = load_model(model_path)

# Step 2: Define the test data directory and generator
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'  # Replace with your test folder path

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input  # Use same preprocessing as training
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Do not shuffle to maintain the order of predictions
)

# Step 3: Predict on the test dataset
test_predictions = model.predict(test_generator)
test_predictions = (test_predictions > 0.5).astype(int).flatten()

# Step 4: Evaluate the model's performance
test_true = test_generator.classes  # True labels from the test dataset

# Classification Report
print("\nClassification Report:")
print(classification_report(test_true, test_predictions, target_names=['Benign', 'Malignant']))

# ROC-AUC Score
roc_auc = roc_auc_score(test_true, model.predict(test_generator).flatten())
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Step 5: Displaying Overall Accuracy
accuracy = np.mean(test_predictions == test_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Found 660 images belonging to 2 classes.
21/21 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      0.75      0.83       360
   Malignant       0.76      0.94      0.84       300

    accuracy                           0.83       660
   macro avg       0.85      0.84      0.83       660
weighted avg       0.85      0.83      0.83       660

21/21 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step
ROC-AUC Score: 0.9356
Test Accuracy: 83.48%


## Hyperparameter Tuning with SAM

In [ ]:
!pip install keras_tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.1 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, Multiply, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive

drive.mount('/content/drive')

# -------------------------------------------
# 1. Define the Spatial Attention Module
# -------------------------------------------
def spatial_attention_module(feature_map, hp):
    """
    A simple Spatial Attention Module (SAM).
    Includes optional hyperparameters that you might want to tune (e.g., dropout).
    """
    # Example hyperparameter for the dropout rate in SAM
    dropout_rate = hp.Float('sam_dropout_rate', min_value=0.2, max_value=0.5, step=0.1)

    # Example hyperparameter for L2 regularization
    l2_reg = hp.Choice('sam_l2_reg', values=[1e-5, 1e-4, 1e-3])

    # Attention map
    attention_map = Conv2D(
        1,
        kernel_size=(3, 3),
        activation='sigmoid',
        padding='same',
        kernel_regularizer=l2(l2_reg)
    )(feature_map)

    attention_map = Dropout(dropout_rate)(attention_map)

    # Multiply feature map by attention map
    enhanced_feature_map = Multiply()([feature_map, attention_map])

    # Optional residual connection
    output = Add()([feature_map, enhanced_feature_map])
    return output

# -------------------------------------------
# 2. Define Model Builder for Keras Tuner
# -------------------------------------------
def model_builder(hp):
    # Base EfficientNetB0
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Optionally unfreeze base model
    base_model.trainable = hp.Boolean('unfreeze_base', default=False)

    # Extract a feature map from one of the blocks
    block_output = base_model.get_layer('block5a_activation').output

    # Insert the Spatial Attention Module
    x = spatial_attention_module(block_output, hp)

    # Global Average Pooling
    x = GlobalAveragePooling2D()(x)

    # Tune the dropout rate for the classifier head
    dropout_rate = hp.Float('dropout_rate', min_value=0.3, max_value=0.6, step=0.1)
    x = Dropout(dropout_rate)(x)

    # Tune the number of units in the Dense layer
    dense_units = hp.Int('dense_units', min_value=64, max_value=256, step=64)
    x = Dense(dense_units, activation='relu')(x)

    # Output layer
    output = Dense(1, activation='sigmoid')(x)

    # Build final model
    model = Model(inputs=base_model.input, outputs=output)

    # Hyperparameter for learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# -------------------------------------------
# 3. Setup Your Data Generators
# -------------------------------------------
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# -------------------------------------------
# 4. Run Keras Tuner (RandomSearch Example)
# -------------------------------------------
tuner = kt.RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=10,  # You can increase this if you have time/resources
    directory='efficientnet_sam_tuning',
    project_name='sam_integration'
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=10,   # Short epochs for the search; adjust as needed
    callbacks=[early_stopping]
)

# -------------------------------------------
# 5. Retrieve Best Model & Final Training
# -------------------------------------------
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hps.values)

best_model = tuner.hypermodel.build(best_hps)

# You can train more epochs using the best hyperparameters
history = best_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# -------------------------------------------
# 6. Evaluate on Test Set
# -------------------------------------------
loss, accuracy = best_model.evaluate(test_generator)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# -------------------------------------------
# 7. Save the Model
# -------------------------------------------
save_path = "/content/drive/MyDrive/FINAL-SAM-HPT-MODEL.h5"  # Adjust as needed
best_model.save(save_path)
print(f"Model saved to: {save_path}")


Trial 10 Complete [00h 28m 25s]
val_accuracy: 0.7817836999893188

Best val_accuracy So Far: 0.8823529481887817
Total elapsed time: 08h 48m 48s
Best Hyperparameters: {'unfreeze_base': True, 'sam_dropout_rate': 0.2, 'sam_l2_reg': 0.0001, 'dropout_rate': 0.5, 'dense_units': 128, 'learning_rate': 0.001}
Epoch 1/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 637s 9s/step - accuracy: 0.7430 - loss: 0.4596 - val_accuracy: 0.5844 - val_loss: 1.6633
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 592s 9s/step - accuracy: 0.8691 - loss: 0.2663 - val_accuracy: 0.5958 - val_loss: 1.1703
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 587s 9s/step - accuracy: 0.9076 - loss: 0.2074 - val_accuracy: 0.7552 - val_loss: 0.5864
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 628s 9s/step - accuracy: 0.9142 - loss: 0.1749 - val_accuracy: 0.7723 - val_loss: 0.5888
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 620s 9s/step - accuracy: 0.9526 - loss: 0.1246 - val_accuracy: 0.8387 - val_loss: 0.4376
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 589s 9s/step - accuracy: 0

Test Loss: 0.2477
Test Accuracy: 0.8985
Model saved to: /content/drive/MyDrive/FINAL-SAM-HPT-MODEL.h5


In [ ]:
import keras_tuner as kt
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from google.colab import drive
import tensorflow as tf

# Mount Google Drive
drive.mount('/content/drive')

# Dataset paths
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'

# Define ImageDataGenerator
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Define the model builder function for KerasTuner
def model_builder(hp):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = hp.Boolean('trainable_base', default=False)  # Hyperparameter to unfreeze base model

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(hp.Float('dropout_rate', min_value=0.3, max_value=0.6, step=0.1))(x)
    x = Dense(
        units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
        activation='relu',
        kernel_initializer='he_normal'
    )(x)
    output = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=output)

    learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Initialize Keras Tuner with RandomSearch
tuner = kt.RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=5,  # Limit trials to 5
    directory='/content/drive/MyDrive/hyperparameter_tuning',
    project_name='efficientnet_tuning'
)

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Search for the best hyperparameters
tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=[early_stopping]
)

# Get the best hyperparameters and build the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {best_hps.values}")

model = tuner.hypermodel.build(best_hps)

# Train the best model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# Evaluate the best model on the test set
test_predictions = model.predict(test_generator)
test_predictions = (test_predictions > 0.5).astype(int).flatten()
test_true = test_generator.classes

# Classification Report
print("\nClassification Report:")
print(classification_report(test_true, test_predictions, target_names=['Benign', 'Malignant']))

# ROC-AUC Score
roc_auc = roc_auc_score(test_true, model.predict(test_generator).flatten())
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Overall Accuracy
accuracy = np.mean(test_predictions == test_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Mounted at /content/drive
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Found 660 images belonging to 2 classes.
Reloading Tuner from /content/drive/MyDrive/hyperparameter_tuning/efficientnet_tuning/tuner0.json
Best Hyperparameters: {'trainable_base': True, 'dropout_rate': 0.3, 'dense_units': 64, 'learning_rate': 0.0005}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


66/66 ━━━━━━━━━━━━━━━━━━━━ 1029s 13s/step - accuracy: 0.8115 - loss: 0.3992 - val_accuracy: 0.5560 - val_loss: 2.1663
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 732s 11s/step - accuracy: 0.9349 - loss: 0.1675 - val_accuracy: 0.5958 - val_loss: 1.3006
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 755s 11s/step - accuracy: 0.9551 - loss: 0.1134 - val_accuracy: 0.6679 - val_loss: 1.3006
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 738s 11s/step - accuracy: 0.9763 - loss: 0.0891 - val_accuracy: 0.7514 - val_loss: 0.8479
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 736s 11s/step - accuracy: 0.9817 - loss: 0.0608 - val_accuracy: 0.7780 - val_loss: 0.8657
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 735s 11s/step - accuracy: 0.9781 - loss: 0.0777 - val_accuracy: 0.8197 - val_loss: 0.7950
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 743s 11s/step - accuracy: 0.9851 - loss: 0.0369 - val_accuracy: 0.8292 - val_loss: 0.9762
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 741s 11s/step - accuracy: 0.9931 - loss: 0.0201 - val_accuracy: 0.8406 - val

# Increased No. or Trials and Epochs for tuning

In [ ]:
import keras_tuner as kt
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from google.colab import drive
import tensorflow as tf

# Mount Google Drive
drive.mount('/content/drive')

# Dataset paths
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'

# Define ImageDataGenerator
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Define the model builder function for KerasTuner
def model_builder(hp):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = hp.Boolean('trainable_base', default=False)  # Hyperparameter to unfreeze base model

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(hp.Float('dropout_rate', min_value=0.3, max_value=0.6, step=0.1))(x)
    x = Dense(
        units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
        activation='relu',
        kernel_initializer='he_normal'
    )(x)
    output = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=output)

    learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Initialize Keras Tuner with RandomSearch
tuner = kt.RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=10,  # Increased max trials
    directory='/content/drive/MyDrive/hyperparameter_tuning',
    project_name='efficientnet_tuning'
)

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Search for the best hyperparameters
tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=10,  # Increased epochs for the tuning process
    callbacks=[early_stopping]
)

# Get the best hyperparameters and build the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {best_hps.values}")

model = tuner.hypermodel.build(best_hps)

# Train the best model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,  # Increased training epochs
    callbacks=[early_stopping]
)

# Save the trained model to Google Drive
model.save('/content/drive/MyDrive/samv2_tuned_model.h5')
print("Model saved as samv2_tuned_model.h5")

# Evaluate the best model on the test set
test_predictions = model.predict(test_generator)
test_predictions = (test_predictions > 0.5).astype(int).flatten()
test_true = test_generator.classes

# Classification Report
print("\nClassification Report:")
print(classification_report(test_true, test_predictions, target_names=['Benign', 'Malignant']))

# ROC-AUC Score
roc_auc = roc_auc_score(test_true, model.predict(test_generator).flatten())
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Overall Accuracy
accuracy = np.mean(test_predictions == test_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Mounted at /content/drive
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Found 660 images belonging to 2 classes.
Reloading Tuner from /content/drive/MyDrive/hyperparameter_tuning/efficientnet_tuning/tuner0.json
Best Hyperparameters: {'trainable_base': True, 'dropout_rate': 0.3, 'dense_units': 64, 'learning_rate': 0.0005}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


66/66 ━━━━━━━━━━━━━━━━━━━━ 1337s 18s/step - accuracy: 0.7548 - loss: 0.4700 - val_accuracy: 0.5541 - val_loss: 2.6349
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 701s 11s/step - accuracy: 0.9202 - loss: 0.1810 - val_accuracy: 0.6129 - val_loss: 2.1244
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 689s 10s/step - accuracy: 0.9694 - loss: 0.0964 - val_accuracy: 0.6641 - val_loss: 1.8765
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 752s 11s/step - accuracy: 0.9725 - loss: 0.0716 - val_accuracy: 0.6319 - val_loss: 2.3216
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 733s 10s/step - accuracy: 0.9735 - loss: 0.0778 - val_accuracy: 0.7742 - val_loss: 0.9581
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 741s 10s/step - accuracy: 0.9839 - loss: 0.0509 - val_accuracy: 0.7913 - val_loss: 1.0866
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 695s 11s/step - accuracy: 0.9825 - loss: 0.0558 - val_accuracy: 0.6964 - val_loss: 1.5571
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 694s 11s/step - accuracy: 0.9812 - loss: 0.0564 - val_accuracy: 0.8463 - val

Model saved as samv2_tuned_model.h5
21/21 ━━━━━━━━━━━━━━━━━━━━ 518s 26s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.89      0.94      0.92       360
   Malignant       0.93      0.86      0.89       300

    accuracy                           0.91       660
   macro avg       0.91      0.90      0.91       660
weighted avg       0.91      0.91      0.91       660

21/21 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step
ROC-AUC Score: 0.9697
Test Accuracy: 90.76%


\## **Hyperparameter tuning SAM with ACO**

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, Multiply, Add
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import random
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
# Define the Spatial Attention Module
def spatial_attention_module(feature_map, dropout_rate, l2_reg):
    attention_map = Conv2D(
        1, kernel_size=(3, 3), activation='sigmoid', padding='same', kernel_regularizer=l2(l2_reg)
    )(feature_map)
    attention_map = Dropout(dropout_rate)(attention_map)
    enhanced_feature_map = Multiply()([feature_map, attention_map])
    output = Add()([feature_map, enhanced_feature_map])
    return output

# Define model function
def build_model(dropout_rate, dense_units, learning_rate, l2_reg):
    print(f"Building model with dropout_rate={dropout_rate}, dense_units={dense_units}, learning_rate={learning_rate}, l2_reg={l2_reg}")
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    block_output = base_model.get_layer('block5a_activation').output
    x = spatial_attention_module(block_output, dropout_rate, l2_reg)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Define the fitness function for ACO
def evaluate_hyperparams(params, train_gen, val_gen):
    print(f"Evaluating hyperparameters: {params}")
    model = build_model(*params)
    history = model.fit(train_gen, validation_data=val_gen, epochs=5, verbose=0)
    max_val_acc = max(history.history['val_accuracy'])
    print(f"Validation Accuracy: {max_val_acc}")
    return max_val_acc

# ACO for Hyperparameter Tuning
def ant_colony_optimization(num_ants, num_generations, train_gen, val_gen):
    print("Starting Ant Colony Optimization...")
    dropout_rates = np.arange(0.2, 0.6, 0.1)
    dense_units = [64, 128, 256]
    learning_rates = [1e-3, 5e-4, 1e-4]
    l2_regs = [1e-5, 1e-4, 1e-3]

    pheromone = np.ones((len(dropout_rates), len(dense_units), len(learning_rates), len(l2_regs)))
    best_params = None
    best_score = 0

    for gen in range(num_generations):
        print(f"Generation {gen+1}/{num_generations}...")
        candidates = []
        scores = []

        for ant in range(num_ants):
            print(f"Ant {ant+1}/{num_ants} exploring...")
            d = random.choices(dropout_rates, weights=pheromone.sum(axis=(1, 2, 3)), k=1)[0]
            du = random.choices(dense_units, weights=pheromone.sum(axis=(0, 2, 3)), k=1)[0]
            lr = random.choices(learning_rates, weights=pheromone.sum(axis=(0, 1, 3)), k=1)[0]
            l2 = random.choices(l2_regs, weights=pheromone.sum(axis=(0, 1, 2)), k=1)[0]
            params = (d, du, lr, l2)
            candidates.append(params)
            score = evaluate_hyperparams(params, train_gen, val_gen)
            scores.append(score)
            if score > best_score:
                best_score = score
                best_params = params
                print(f"New Best Parameters Found: {best_params} with Score: {best_score}")

        # Update pheromones
        for i, (d, du, lr, l2) in enumerate(candidates):
            d_idx = np.where(dropout_rates == d)[0][0]
            du_idx = dense_units.index(du)
            lr_idx = learning_rates.index(lr)
            l2_idx = l2_regs.index(l2)
            pheromone[d_idx, du_idx, lr_idx, l2_idx] += scores[i]
            pheromone *= 0.9  # Evaporation

    return best_params

# Setup Data Generators
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'

datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet.preprocess_input, validation_split=0.2)
train_generator = datagen.flow_from_directory(train_data_dir, target_size=(224, 224), batch_size=32, class_mode='binary', subset='training')
val_generator = datagen.flow_from_directory(train_data_dir, target_size=(224, 224), batch_size=32, class_mode='binary', subset='validation')

# Run ACO
best_hyperparams = ant_colony_optimization(num_ants=7, num_generations=4, train_gen=train_generator, val_gen=val_generator)
print(f"Best Hyperparameters: {best_hyperparams}")

# Train final model with best hyperparameters
final_model = build_model(*best_hyperparams)
print("Training final model with best hyperparameters...")
early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)
history = final_model.fit(train_generator, validation_data=val_generator, epochs=20, callbacks=[early_stopping])

# Save final model
print("Saving final model...")
final_model.save('/content/drive/MyDrive/FINAL-SAM-ACO-MODEL2.h5')
print("Model saved successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Starting Ant Colony Optimization...
Generation 1/4...
Ant 1/7 exploring...
Evaluating hyperparameters: (np.float64(0.4000000000000001), 256, 0.0005, 0.0001)
Building model with dropout_rate=0.4000000000000001, dense_units=256, learning_rate=0.0005, l2_reg=0.0001
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Validation Accuracy: 0.8197343349456787
New Best Parameters Found: (np.float64(0.4000000000000001), 256, 0.0005, 0.0001) with Score: 0.8197343349456787
Ant 2/7 exploring...
Evaluating hyperparameters: (np.float64(0.30000000000000004), 128, 0.001, 1e-05)
Building model with dropout_rate=0.30000000000000004, dense_units=128, learning_rate=0.001, l2_reg=1e-05
Validation Accuracy: 0.8178368210792542
Ant 3/7 exploring...
Evaluating hyperparameters: (np.float64(0.5000000000000001), 128, 0.0005, 1e-05)
Building model with dropout_rate=0.5000000000000001, dense_units=128, learning_rate=0.0005, l2_reg=1e-05
Validation Accuracy: 0.8083491325378418
Ant 4/7 exploring...
Evaluating hyperparameters: (np.float64(0.2), 256, 0.001, 0.0001)
Building model with dropout_rate=0.2, dense_units=256, learning_rate=0.001, l2_reg=0.0001
Validation Accuracy: 0.8216318488121033
New Best Parameters Found: (np.float64(0.2), 256, 0.001, 0.0001) with Score: 0.8216318488121033
Ant 5/7 exploring...
Evaluating hyperpara

Saving final model...
Model saved successfully!


In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

drive.mount('/content/drive')


# Step 1: Load the trained model
model_path = '/content/drive/MyDrive/FINAL-SAM-ACO-MODEL.h5'
model = load_model(model_path)

# Step 2: Define the test data directory and generator
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'  # Replace with your test folder path

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input  # Use same preprocessing as training
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Do not shuffle to maintain the order of predictions
)

# Step 3: Predict on the test dataset
test_predictions = model.predict(test_generator)
test_predictions = (test_predictions > 0.5).astype(int).flatten()

# Step 4: Evaluate the model's performance
test_true = test_generator.classes  # True labels from the test dataset

# Classification Report
print("\nClassification Report:")
print(classification_report(test_true, test_predictions, target_names=['Benign', 'Malignant']))

# ROC-AUC Score
roc_auc = roc_auc_score(test_true, model.predict(test_generator).flatten())
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Step 5: Displaying Overall Accuracy
accuracy = np.mean(test_predictions == test_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Found 660 images belonging to 2 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


21/21 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step

Classification Report:
              precision    recall  f1-score   support

      Benign       0.90      0.82      0.86       360
   Malignant       0.80      0.89      0.85       300

    accuracy                           0.85       660
   macro avg       0.85      0.86      0.85       660
weighted avg       0.86      0.85      0.85       660

21/21 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step
ROC-AUC Score: 0.9506
Test Accuracy: 85.30%


# Hyerparameter Tuning SAM using Sun Flower Optimzation

> Add blockquote



In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, Multiply, Add
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import random
from google.colab import drive

drive.mount('/content/drive')

# Define the Spatial Attention Module
def spatial_attention_module(feature_map, dropout_rate, l2_reg):
    attention_map = Conv2D(
        1, kernel_size=(3, 3), activation='sigmoid', padding='same', kernel_regularizer=l2(l2_reg)
    )(feature_map)
    attention_map = Dropout(dropout_rate)(attention_map)
    enhanced_feature_map = Multiply()([feature_map, attention_map])
    output = Add()([feature_map, enhanced_feature_map])
    return output

# Define model function
def build_model(dropout_rate, dense_units, learning_rate, l2_reg):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    block_output = base_model.get_layer('block5a_activation').output
    x = spatial_attention_module(block_output, dropout_rate, l2_reg)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Define fitness function
def evaluate_hyperparams(params, train_gen, val_gen):
    model = build_model(*params)
    history = model.fit(train_gen, validation_data=val_gen, epochs=5, verbose=0)
    max_val_acc = max(history.history['val_accuracy'])
    return max_val_acc

# Sunflower Optimization Algorithm for Hyperparameter Tuning
def sunflower_optimization(num_sunflowers, num_generations, train_gen, val_gen):
    dropout_rates = np.arange(0.2, 0.6, 0.1)
    dense_units = [64, 128, 256]
    learning_rates = [1e-3, 5e-4, 1e-4]
    l2_regs = [1e-5, 1e-4, 1e-3]

    sunflowers = [
        (random.choice(dropout_rates), random.choice(dense_units), random.choice(learning_rates), random.choice(l2_regs))
        for _ in range(num_sunflowers)
    ]

    best_params = None
    best_score = 0

    for gen in range(num_generations):
        print(f"Generation {gen+1}/{num_generations}...")
        scores = [evaluate_hyperparams(params, train_gen, val_gen) for params in sunflowers]

        best_idx = np.argmax(scores)
        if scores[best_idx] > best_score:
            best_score = scores[best_idx]
            best_params = sunflowers[best_idx]
            print(f"New Best Parameters Found: {best_params} with Score: {best_score}")

        # Update sunflowers based on fitness (best solutions move towards better ones)
        new_sunflowers = []
        for i in range(num_sunflowers):
            new_sunflower = [
                max(0.2, min(0.6, sunflowers[i][0] + random.uniform(-0.05, 0.05))),
                random.choice(dense_units),
                random.choice(learning_rates),
                random.choice(l2_regs)
            ]
            new_sunflowers.append(tuple(new_sunflower))
        sunflowers = new_sunflowers

    return best_params

# Setup Data Generators
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir = '/content/drive/MyDrive/kaggle_dataset_test'

datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet.preprocess_input, validation_split=0.2)
train_generator = datagen.flow_from_directory(train_data_dir, target_size=(224, 224), batch_size=32, class_mode='binary', subset='training')
val_generator = datagen.flow_from_directory(train_data_dir, target_size=(224, 224), batch_size=32, class_mode='binary', subset='validation')

# Run Sunflower Optimization
best_hyperparams = sunflower_optimization(num_sunflowers=7, num_generations=4, train_gen=train_generator, val_gen=val_generator)
print(f"Best Hyperparameters: {best_hyperparams}")

# Train final model with best hyperparameters
final_model = build_model(*best_hyperparams)
early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)
history = final_model.fit(train_generator, validation_data=val_generator, epochs=20, callbacks=[early_stopping])

# Save final model
final_model.save('/content/drive/MyDrive/FINAL-SAM-SFO-MODEL.h5')
print("Model saved successfully!")


Mounted at /content/drive
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Generation 1/4...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


New Best Parameters Found: (np.float64(0.4000000000000001), 64, 0.001, 0.0001) with Score: 0.8273244500160217
Generation 2/4...


# **Hyperparameter Tuning SAM Cat Mouse Optimization**

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, Multiply, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import random
from google.colab import drive

# -----------------------------
# 1. Drive Mount
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 2. Define Spatial Attention Module
# -----------------------------
def spatial_attention_module(feature_map, dropout_rate, l2_reg):
    """
    A simple Spatial Attention Module (SAM).
    """
    attention_map = Conv2D(
        1, kernel_size=(3, 3), activation='sigmoid', padding='same', kernel_regularizer=l2(l2_reg)
    )(feature_map)
    attention_map = Dropout(dropout_rate)(attention_map)
    enhanced_feature_map = Multiply()([feature_map, attention_map])
    output = Add()([feature_map, enhanced_feature_map])  # Residual connection
    return output

# -----------------------------
# 3. Define Model-Building Function
# -----------------------------
def build_model(dropout_rate, dense_units, learning_rate, l2_reg):
    """
    Builds an EfficientNetB0-based model with a Spatial Attention Module (SAM) at block5a_activation.
    """
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # Freeze base model layers (just as an example; you can unfreeze if desired)
    base_model.trainable = False

    # Extract feature map from block5a_activation
    block_output = base_model.get_layer('block5a_activation').output

    # Insert SAM
    x = spatial_attention_module(block_output, dropout_rate, l2_reg)

    # Classification Head
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# -----------------------------
# 4. Fitness Function
# -----------------------------
def evaluate_hyperparams(params, train_gen, val_gen, short_epochs=8):
    """
    Given hyperparameters, train a small number (short_epochs) of epochs and return best validation accuracy.
    """
    model = build_model(*params)
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=short_epochs,  # short training for quick evaluation
        verbose=0
    )
    max_val_acc = max(history.history['val_accuracy'])
    return max_val_acc

# -----------------------------
# 5. Cat and Mouse Optimization
# -----------------------------
def cat_mouse_optimization(
    num_cats,
    num_mice,
    num_generations,
    train_gen,
    val_gen,
    short_epochs=8
):
    """
    A simple illustrative 'Cat and Mouse' metaheuristic:
      - We maintain a population of cats and mice (hyperparams).
      - Each generation, we evaluate fitness (val accuracy).
      - The best solution is a 'target.'
      - Cats move towards the best solution; mice move away, with random perturbations.
    """

    # Expanded Hyperparam search space
    dropout_rates  = np.arange(0.2, 0.75, 0.1)   # [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
    dense_units    = [64, 128, 256, 512]
    learning_rates = [1e-3, 5e-4, 1e-4, 1e-5]
    l2_regs        = [1e-6, 1e-5, 1e-4, 1e-3]

    def random_hyperparams():
        return (
            float(random.choice(dropout_rates)),
            random.choice(dense_units),
            random.choice(learning_rates),
            random.choice(l2_regs)
        )

    # Initialize population
    cats = [random_hyperparams() for _ in range(num_cats)]
    mice = [random_hyperparams() for _ in range(num_mice)]

    best_params = None
    best_score = 0.0

    for gen in range(num_generations):
        print(f"\n--- Generation {gen+1}/{num_generations} ---")

        # Evaluate all cats and mice
        cat_scores = [evaluate_hyperparams(c, train_gen, val_gen, short_epochs=short_epochs) for c in cats]
        mouse_scores = [evaluate_hyperparams(m, train_gen, val_gen, short_epochs=short_epochs) for m in mice]

        # Find best among cats + mice
        all_scores  = cat_scores + mouse_scores
        all_solvers = cats + mice
        current_best_idx = np.argmax(all_scores)
        current_best_score = all_scores[current_best_idx]
        current_best_params = all_solvers[current_best_idx]

        # If we found a new best overall, update
        if current_best_score > best_score:
            best_score = current_best_score
            best_params = current_best_params
            print(f"New Best Found: Params={best_params}, ValAcc={best_score:.4f}")
        else:
            print(f"No improvement. Best so far: Params={best_params}, ValAcc={best_score:.4f}")

        # "Cat Move": move cats closer to best_params
        new_cats = []
        for i, c in enumerate(cats):
            # c is (dropout_rate, dense_units, lr, l2_reg)
            current_dr, current_du, current_lr, current_l2 = c
            best_dr, best_du, best_lr, best_l2 = best_params

            # Move dropout rate closer to best's dropout rate
            new_dr = current_dr + (best_dr - current_dr) * random.uniform(0.1, 0.3)
            new_dr = float(np.clip(new_dr, dropout_rates[0], dropout_rates[-1]))

            # For discrete ones, let's randomly pick current or best
            new_du = random.choice([current_du, best_du])
            new_lr = random.choice([current_lr, best_lr])
            new_l2 = random.choice([current_l2, best_l2])

            new_cats.append((new_dr, new_du, new_lr, new_l2))

        # "Mouse Move": mice move away from best_params
        new_mice = []
        for i, m in enumerate(mice):
            current_dr, current_du, current_lr, current_l2 = m
            best_dr, best_du, best_lr, best_l2 = best_params

            # Move dropout away from best by a fraction
            direction = np.sign(current_dr - best_dr)  # +1 or -1
            if direction == 0:
                # If it's exactly the same, random push
                direction = random.choice([-1, 1])
            new_dr = current_dr + direction * random.uniform(0.1, 0.3)
            new_dr = float(np.clip(new_dr, dropout_rates[0], dropout_rates[-1]))

            # For discrete ones, let's randomly pick from current or re-random
            # to maintain diversity:
            new_du = random.choice([current_du, random.choice(dense_units)])
            new_lr = random.choice([current_lr, random.choice(learning_rates)])
            new_l2 = random.choice([current_l2, random.choice(l2_regs)])

            new_mice.append((new_dr, new_du, new_lr, new_l2))

        cats = new_cats
        mice = new_mice

    return best_params

# -----------------------------
# 6. Data Generators
# -----------------------------
train_data_dir = '/content/drive/MyDrive/kaggle_dataset_train'
test_data_dir  = '/content/drive/MyDrive/kaggle_dataset_test'

BATCH_SIZE = 64  # Increased batch size for faster iteration if GPU memory allows

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=train_data_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# Test data generator (no validation split)
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    directory=test_data_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# -----------------------------
# 7. Run Cat and Mouse Optimization
# -----------------------------
best_hyperparams = cat_mouse_optimization(
    num_cats=8,         # slightly increased, can go higher if you have more GPU
    num_mice=8,
    num_generations=6,  # can increase further if you want more thorough search
    train_gen=train_generator,
    val_gen=val_generator,
    short_epochs=8      # short training epochs for each trial
)

print(f"\nBest Hyperparameters Found: {best_hyperparams}")

# -----------------------------
# 8. Train Final Model
# -----------------------------
final_model = build_model(*best_hyperparams)

# Increase final training epochs and patience
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = final_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,  # More epochs for final training
    callbacks=[early_stopping]
)

# -----------------------------
# 9. Evaluate and Save Model
# -----------------------------
loss, acc = final_model.evaluate(test_generator)
print(f"Final Model Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")

save_path = '/content/drive/MyDrive/FINAL-SAM-CatMouseOpt-MODEL.h5'
final_model.save(save_path)
print(f"Model saved to: {save_path}")


Mounted at /content/drive
Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Found 660 images belonging to 2 classes.

--- Generation 1/6 ---
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


New Best Found: Params=(0.4000000000000001, 64, 0.001, 1e-06), ValAcc=0.8349

--- Generation 2/6 ---
No improvement. Best so far: Params=(0.4000000000000001, 64, 0.001, 1e-06), ValAcc=0.8349

--- Generation 3/6 ---
